In [1]:
-- Gold layer: monthly revenue and order volume trend
-- Business Question 1

CREATE OR REPLACE TABLE gold_monthly_revenue AS
SELECT 
    YEAR(o.order_purchase_timestamp) AS order_year,
    MONTH(o.order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.price) AS total_revenue
FROM silver_orders AS o 
JOIN silver_order_items AS oi ON o.order_id = oi.order_id
GROUP BY YEAR(o.order_purchase_timestamp),MONTH(o.order_purchase_timestamp)
ORDER BY order_year, order_month

StatementMeta(, 9c26d8e9-f8f3-4e33-b210-440564a69200, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
-- Gold layer: revenue by product category
-- Business Question 2

CREATE OR REPLACE TABLE gold_category_performance AS
SELECT
    ct.product_category_name_english,
    SUM(oi.price) AS total_revenue
FROM silver_order_items AS oi
JOIN silver_products AS p ON oi.product_id = p.product_id
JOIN silver_category_translation AS ct ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY total_revenue DESC


StatementMeta(, 9c26d8e9-f8f3-4e33-b210-440564a69200, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
-- Gold layer: average delivery performance per seller
-- Business Question 3


CREATE OR REPLACE TABLE gold_delivery_performance AS
SELECT 
    s.seller_id,
    AVG(DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date)) AS avg_delivery_days
FROM silver_sellers AS s
JOIN silver_order_items AS oi ON s.seller_id = oi.seller_id
JOIN silver_orders AS o ON oi.order_id = o.order_id
GROUP BY s.seller_id
ORDER BY avg_delivery_days ASC


StatementMeta(, 9c26d8e9-f8f3-4e33-b210-440564a69200, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
-- Gold layer: relationship between delivery delay and review score
-- Business Question 4

CREATE OR REPLACE TABLE gold_satisfaction_driver AS
SELECT
    r.review_score,
    AVG(DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date)) AS avg_delivery_days
FROM silver_reviews r
JOIN silver_orders o ON r.order_id = o.order_id
GROUP BY r.review_score
ORDER BY r.review_score DESC
   

StatementMeta(, 9c26d8e9-f8f3-4e33-b210-440564a69200, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
-- Gold layer: Pareto analysis -- which sellers drive 80% of revenue
-- Business Question 5

CREATE OR REPLACE TABLE gold_pareto_analysis AS
WITH SellerRevenue AS (
    SELECT
        oi.seller_id,
        SUM(oi.price) AS total_revenue
    FROM silver_order_items AS oi
    JOIN silver_orders AS  o ON oi.order_id = o.order_id
    GROUP BY oi.seller_id )
SELECT 
    seller_id,
    total_revenue,
    SUM(total_revenue) OVER (ORDER BY total_revenue DESC) AS running_total,
    SUM(total_revenue) OVER () AS grand_total,
    SUM(total_revenue) OVER (ORDER BY total_revenue DESC) / SUM(total_revenue) OVER () * 100 AS cumulative_percentage
FROM SellerRevenue
ORDER BY total_revenue DESC

StatementMeta(, 9c26d8e9-f8f3-4e33-b210-440564a69200, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>